In [3]:
!pip install word2number

  Preparing metadata (setup.py) ... done
  Created wheel for word2number: filename=word2number-1.1-py3-none-any.whl size=5657 sha256=3369b4750be61e2b7dfc14768cb4e587c61cda293b85676ce215345add7112d9
  Stored in directory: /root/.cache/pip/wheels/aa/2b/71/ede4a3c3520a8374778c16c345e0e78e25d1ffe38f4c5c1e8a
Successfully built word2number


In [12]:
from google.colab import files
from word2number import w2n
import numpy as np
import re
import pandas as pd
import io

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(io.BytesIO(uploaded[file_name]))
df.head()

Saving Messy Animal Data - Animal Dataset (1).csv to Messy Animal Data - Animal Dataset (1) (3).csv


,ID,Animal,Height (cm),Weight (kg),Color,Lifespan (years),Diet,Habitat,Predators,Average Speed (km/h),Countries Found,Conservation Status,Family,Gestation Period (days),Top Speed (km/h),Social Structure,Offspring per Birth
0,1,Aardvark,105-130,40-65,Grey,20-30,NaN,"Savannas, Grasslands","Lions, Hyenas",40,Africa,Least Concern,Orycteropodidae,210-240,40,Solitary,1
1,2,Aardwolf,40-50,8-14,Yellow-brown,10-12,NaN,"Grasslands, Savannas","Lions, Leopards",24-30,Eastern and Southern Africa,Least Concern,Hyaenidae,90,40,Solitary,2-5
2,3,African Elephant,270-310,2700-6000,Grey,60-70,Herbivore,"Savannah, Forest","Lions, Hyenas",25,Africa,Vulnerable,Elephantidae,640-660,40,Herd-based,1
3,4,African Lion,80-110,120-250,Tan,10-14,Carnivore,"Grasslands, Savannas","Hyenas, Crocodiles",58,Africa,Vulnerable,Felidae,98-105,80,Group-based,2-4 (usually)
4,5,African Wild Dog,75-80,18-36,Multicolored,10-12,Carnivore,Savannahs,"Lions, Hyenas",fifty-six,Sub-Saharan Africa,Endangered,Canidae,70,56,Group-based,10-12


In [15]:
df.info()
print(df['Animal'].unique())
print(df['Average Speed (km/h)'].unique())
print(df['Diet'].unique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   ID                       205 non-null    int64 
 1   Animal                   192 non-null    object
 2   Height (cm)              205 non-null    object
 3   Weight (kg)              205 non-null    object
 4   Color                    205 non-null    object
 5   Lifespan (years)         205 non-null    object
 6   Diet                     196 non-null    object
 7   Habitat                  205 non-null    object
 8   Predators                205 non-null    object
 9   Average Speed (km/h)     205 non-null    object
 10  Countries Found          205 non-null    object
 11  Conservation Status      205 non-null    object
 12  Family                   205 non-null    object
 13  Gestation Period (days)  205 non-null    object
 14  Top Speed (km/h)         205 non-null    o

In [16]:
# Keep columns we are considering and rename for ease of reference
df = df.rename(columns={
    'Animal': 'name',
    'Average Speed (km/h)': 'speed',
    'Diet': 'diet'
})
df = df[['name', 'speed', 'diet']].copy()

# Trimming Whitespace
for col in ['name', 'speed', 'diet']:
    df[col] = df[col].astype('string').str.strip()

# Removing rows with missing values (including NA and other words)
df['speed'] = df['speed'].replace(['Not Applicable', 'Varies'], np.nan)
df = df.dropna(subset=['name', 'speed', 'diet'])

def clean_speed(val):
    # Unnecessary asides
    val = re.sub(r'\(.*?\)', '', val).strip()

    # Range to average
    if re.fullmatch(r'\d+(\.\d+)?\s*-\s*\d+(\.\d+)?', val):
        lo, hi = (float(p) for p in val.split('-'))
        return (lo + hi) / 2

    # Plain number
    try:
        return float(val)
    except ValueError:
        pass

    # Spelled out number
    try:
        return float(w2n.word_to_num(val))
    except ValueError:
        return np.nan

df['speed'] = df['speed'].apply(clean_speed)
df = df.dropna(subset=['speed'])

# Keep only Carnivore, Herbivore, Omnivore
df['diet'] = df['diet'].str.lower()
df = df[df['diet'].isin(['carnivore', 'herbivore', 'omnivore'])]

# Animal corrections
df['name'] = df['name'].str.replace('Ã¡', 'á')
df['name'] = df['name'].replace('Sumatran Rhino', 'Sumatran Rhinoceros')
df = df.drop_duplicates(subset='name')

df = df.reset_index(drop=True)
df

,name,speed,diet
0,African Elephant,25.0,herbivore
1,African Lion,58.0,carnivore
2,African Wild Dog,56.0,carnivore
3,Alpine Ibex,60.0,herbivore
4,American Bison,48.0,herbivore
...,...,...,...
126,Wombat,20.0,herbivore
127,Yak,24.0,herbivore
128,Yellow-Eyed Penguin,25.0,carnivore
129,Zebra,25.0,herbivore


In [19]:
df.to_csv('Shubh Varshney - Cleaned Animal Data.csv')